# Inicializando

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, regexp_replace, length, isnull

# Lendo dados da Tabela Bronze

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')
df.display()

# Limpeza para adequar a coluna CID aos dados da tabela crm_customers

In [0]:
df = (
    df.withColumn('CID', regexp_replace('CID', '-', ''))
)

df.limit(10).display()

# Verificando integridade da coluna CID

In [0]:
df.select(length(col('CID'))).distinct().display()

# Verificando valores da coluna CNTRY

In [0]:
df.select(trim('CNTRY')).distinct().display()

# Normalizando valores da coluna CNTRY

In [0]:
df = (
    df.withColumn(
        'CNTRY',
        F.when(trim(col('CNTRY')).isin('US', 'USA'), 'United States')
        .when(trim(col('CNTRY')) == 'DE', 'Germany')
        .when(trim(col('CNTRY')) == '', 'n/a')
        .when(isnull(trim(col('CNTRY'))), 'n/a')
        .otherwise(col('CNTRY'))
    )
)

df.select(trim('CNTRY')).distinct().display()


# Renomeando colunas

In [0]:
RENAME_MAP = {
    'CID': 'customer_id',
    'CNTRY': 'country'
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

# Criação de Tabela Silver

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customer_location')

# Verificação da Tabela Final


In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10;